# 🍳 Recipe Generator – Fine-tuning GPT-2

In this notebook, we fine-tune **GPT-2** on a dataset of cooking recipes, aiming to train a model capable of generating new, well-structured recipes from just a given title.

We chose GPT-2 for two key reasons:

- **🖥️ Computational Efficiency**  
  The base version of GPT-2 (117M parameters) is relatively lightweight compared to larger models. This makes it a practical choice for full fine-tuning on environments like Google Colab or Kaggle, without running into memory or time limitations.

- **📈 Strong Performance on Small Datasets**  
  Due to its moderate size, GPT-2 is more responsive to smaller-scale training. Even with a limited set of recipes, we can expect a significant drop in training loss and a clear improvement in the generated outputs—especially in terms of coherence, relevance, and formatting.


## ⚙️ Setup
First, we install and load the required libraries.

In [ ]:
!pip install transformers datasets torch accelerate pandas tqdm rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b25a6bde7fb6e9b602ef534a7c9c167011e2098a0eaa412180afc95fe4c78205
  

In [ ]:
!pip install evaluate nltk bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from transformers import TextDataset, DataCollatorForLanguageModeling
from datasets import Dataset

## 📦 Load and Prepare Data

Let's load the recipes data and format it for training.

### Loading data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/rcp-nlp/simo

/content/drive/.shortcut-targets-by-id/1ii6eI76RNWTvQduHWw9n3IAzswUbvxRG/rcp-nlp/simo


In [ ]:
import time
import ast
import random
import pandas as pd

random.seed(42)

SAMPLE_SIZE = 1000000

def load_data(filepath, sample_size=None):
    """
    Load the RecipeNLG dataset from CSV

    Args:
        filepath: Path to the full_dataset.csv
        sample_size: Number of recipes to sample (None to load all)

    Returns:
        pandas DataFrame with the recipe data
    """
    start_time = time.time()

    # Define the converters to parse stringified lists
    converters = {
        'ingredients': ast.literal_eval,
        'directions': ast.literal_eval,
        'ner': ast.literal_eval
    }

    # For a quick analysis, we can load a sample of the data
    if sample_size is not None:
        # Count the total number of lines in the file
        with open(filepath, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)

        # Generate random line indices to sample
        skip_indices = sorted(random.sample(range(1, line_count), line_count - sample_size - 1))
        df = pd.read_csv(filepath, skiprows=skip_indices, converters=converters)
    else:
        # Load the full dataset
        df = pd.read_csv(filepath, converters=converters)

    # Ensure proper types
    # df['id'] = df['id'].astype(int)

    print(f"Data loaded in {time.time() - start_time:.2f} seconds")
    print(f"Dataset shape: {df.shape}")

    return df

# Try to load a sample of the data for analysis
try:
    # Adjust this path to where you have stored the dataset in Colab
    DATASET_PATH = '/content/drive/MyDrive/rcp-nlp/full_dataset.csv'

    # For initial analysis, let's use a sample of 10,000 recipes
    # This will be much faster while still providing meaningful insights
    df = load_data(DATASET_PATH, sample_size=SAMPLE_SIZE)

    # If you want to load the full dataset (warning: will take time and memory)
    # df = load_data(DATASET_PATH)


except FileNotFoundError:
    # If the dataset is not already in Colab, upload it
    from google.colab import files

    print("Please upload the full_dataset.csv file")


Data loaded in 84.48 seconds
Dataset shape: (999998, 7)


In [ ]:
df

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[1 c. firmly packed brown sugar, 1/2 c. evapor...","[In a heavy 2-quart saucepan, mix brown sugar,...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,2,Creamy Corn,"[2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. c...","[In a slow cooker, combine all ingredients. Co...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
2,4,Reeses Cups(Candy),"[1 c. peanut butter, 3/4 c. graham cracker cru...",[Combine first four ingredients and press in 1...,www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."
3,7,Scalloped Corn,"[1 can cream-style corn, 1 can whole kernel co...","[Mix together both cans of corn, crackers, egg...",www.cookbooks.com/Recipe-Details.aspx?id=876969,Gathered,"[""cream-style corn"", ""whole kernel corn"", ""cra..."
4,10,Double Cherry Delight,"[1 (17 oz.) can dark sweet pitted cherries, 1/...","[Drain cherries, measuring syrup., Cut cherrie...",www.cookbooks.com/Recipe-Details.aspx?id=703381,Gathered,"[""dark sweet pitted cherries"", ""ginger ale"", ""..."
...,...,...,...,...,...,...,...
999991,2231129,Baked Praline Ice Cream Cake,"[1 cup packed brown sugar, 12 cup sour cream, ...","[Set oven to 350 degrees /Grease a 13"" x 9"" ba...",www.food.com/recipe/baked-praline-ice-cream-ca...,Recipes1M,"[""brown sugar"", ""sour cream"", ""butter"", ""butte..."
999992,2231131,Roasted Garlicpea Puree on Sourdough Croutes,"[1 sourdough baguette, 1/4 cup extra-virgin ol...","[Preheat oven to 400F., Diagonally cut twenty-...",www.epicurious.com/recipes/food/views/roasted-...,Recipes1M,"[""baguette"", ""extra-virgin olive oil"", ""garlic..."
999993,2231133,Basil Chicken Parmigiana,"[8 ounces uncooked rotini pasta, 24 ounces chi...",[Prepare pasta according to package directions...,www.food.com/recipe/basil-chicken-parmigiana-2...,Recipes1M,"[""rotini pasta"", ""chicken breasts"", ""Italian b..."
999994,2231134,Cheese-and-Salmon Quesadilla,"[2 can salmon, 1 c. Monterey Jack cheese, 4 fl...","[In a bowl, stir together two 6-oz., cans salm...",www.delish.com/recipefinder/cheese-and-salmon-...,Recipes1M,"[""salmon"", ""cheese"", ""flour tortilla"", ""green ..."


In [ ]:
# convert the name of first column to id
df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
recipes = df.drop(columns=['link','source','NER']).to_dict(orient='records')


### Format Data for Training

We now format each recipe as a text that the model can learn from and save them in a txt file for training.

In [ ]:
# Format the recipes as text
def format_recipe_for_fine_tuning(recipe):
    title = recipe['title']

    prompt = f"Generate a new recipe with this TITLE: {title}\n\nTITLE: {title}\n\nINGREDIENTS:\n"
    prompt += "\n".join(f"- {ingredient}" for ingredient in recipe['ingredients'])
    prompt += "\n\nDIRECTIONS:\n"
    prompt += "\n".join(f"{i+1}. {step}" for i, step in enumerate(recipe['directions']))
    prompt += "\n\n"

    return prompt

# Create training texts
training_texts = [format_recipe_for_fine_tuning(recipe) for recipe in recipes]

# Print a sample to verify
print(training_texts[0])

Generate a new recipe with this TITLE: No-Bake Nut Cookies

TITLE: No-Bake Nut Cookies

INGREDIENTS:
- 1 c. firmly packed brown sugar
- 1/2 c. evaporated milk
- 1/2 tsp. vanilla
- 1/2 c. broken nuts (pecans)
- 2 Tbsp. butter or margarine
- 3 1/2 c. bite size shredded rice biscuits

DIRECTIONS:
1. In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.
2. Stir over medium heat until mixture bubbles all over top.
3. Boil and stir 5 minutes more. Take off heat.
4. Stir in vanilla and cereal; mix well.
5. Using 2 teaspoons, drop and shape into 30 clusters on wax paper.
6. Let stand until firm, about 30 minutes.




In [ ]:
# Save the training data to a text file
with open('recipes_train.txt', 'w') as f:
    for text in training_texts:
        f.write(text + '\n')

## Load and initialize the model and the Tokenizer

We now load GPT-2 model and its tokenizer...

In [ ]:
# Initialize tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Add padding token to tokenizer
tokenizer.pad_token = tokenizer.eos_token

... and this function to generate a new recipe from a title.

In [ ]:
def generate_structured_recipe(title, model, tokenizer, max_length=1024):
    device = model.device

    # Prompt nel formato su cui è stato fatto fine-tuning
    prompt = f"Generate a new recipe with this TITLE: {title}\n\nINGREDIENTS:\n"

    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    output = model.generate(
        inputs['input_ids'],
        min_length=150,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text


## 📊 Pre-trained Model Evaluation




We begin by evaluating the base **GPT-2** model *without any fine-tuning*, to establish a baseline. This allows us to understand how well the pretrained model performs on the task of recipe generation using only its general language knowledge.

We'll assess the generated recipes through a combination of **automated metrics** and **human evaluation**:

- **Semantic similarity**: computed using SBERT cosine similarity between the generated recipe and a ground-truth reference.
- **ROUGE and METEOR** scores: to quantify overlap in terms of n-grams and content between the generated and reference texts.
- **Qualitative analysis**: to evaluate the overall coherence, format adherence, and plausibility of the recipes.

Later in the notebook, we will compare these results with those obtained after fine-tuning, to evaluate the improvements in quality, structure, and relevance introduced by the training process.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import evaluate

# Load evaluation metrics from HuggingFace
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

def evaluate_generated_recipe(title, reference_text, model, tokenizer):
    # Step 1: Generate the recipe
    generated_text = generate_structured_recipe(title, model, tokenizer)

    # Step 2: Semantic similarity with SBERT
    sbert = SentenceTransformer('all-MiniLM-L6-v2')
    emb_ref = sbert.encode(reference_text, convert_to_tensor=True)
    emb_gen = sbert.encode(generated_text, convert_to_tensor=True)
    semantic_score = float(util.pytorch_cos_sim(emb_ref, emb_gen))

    # Step 3: ROUGE (focus only on rouge1)
    rouge_result = rouge.compute(
        predictions=[generated_text],
        references=[reference_text],
        use_stemmer=True
    )
    rouge1_f1 = rouge_result["rouge1"]

    # Step 4: METEOR
    meteor_result = meteor.compute(
        predictions=[generated_text],
        references=[reference_text]
    )
    meteor_score_val = meteor_result["meteor"]

    # Output nicely
    print("=" * 100)
    print(f"🍽 TITLE: {title}\n")
    print("📜 REFERENCE RECIPE:\n", reference_text[:2000], "...\n")
    print("-" * 100)
    print("🧠 GENERATED RECIPE:\n", generated_text[:2000], "...\n")
    print("-" * 100)
    print(f"🔍 Semantic similarity (SBERT cosine): {semantic_score:.4f}")
    print(f"📏 ROUGE-1 F1 score: {rouge1_f1:.4f}")
    print(f"⭐ METEOR score: {meteor_score_val:.4f}")
    print("=" * 100)

    return {
        "title": title,
        "semantic_similarity": semantic_score,
        "rouge1_f1": rouge1_f1,
        "meteor": meteor_score_val,
        "generated_text": generated_text,
        "reference_text": reference_text
    }


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
eval_selected_recipes_id = [375, 6808, 6070]

results = []

for id in eval_selected_recipes_id:
    title = recipes[id]['title']
    reference_text = format_recipe_for_fine_tuning(recipes[id])

    results.append(evaluate_generated_recipe(title, reference_text, model, tokenizer))


🍽 TITLE: Date Rolled Cookies

📜 REFERENCE RECIPE:
 Generate a new recipe with this TITLE: Date Rolled Cookies

TITLE: Date Rolled Cookies

INGREDIENTS:
- 1 c. butter
- 1 c. white sugar
- 1 c. brown sugar
- 1 tsp. salt
- 3 beaten eggs
- 1 Tbsp. vanilla
- 4 c. flour
- 1 1/2 tsp. soda
- 1 Tbsp. cold water

DIRECTIONS:
1. Mix the flour and soda together.
2. Roll 1/2 dough on floured board into a rectangle, 1/2-inch thick.
3. Spread filling all over top (then sprinkle with chopped English walnuts, if desired). Roll like a jelly roll and place in refrigerator.
4. When firm, slice and bake 10 to 12 minutes at 350° or 10 minutes at 375° (top of cookie may be sprinkled with sugar and top with 1/2 walnut before baking).

 ...

----------------------------------------------------------------------------------------------------
🧠 GENERATED RECIPE:
 Generate a new recipe with this TITLE: Date Rolled Cookies

INGREDIENTS:
 (1) 1/2 cup all purpose flour, or 2 tablespoons plus more for grating and bak

### Qualitative analysis
Although the language used in the generated text is generally relevant to the context of food and cooking, and the model occasionally understands that it should provide ingredients and instructions, the overall recipe structure is poorly defined and lacks clear separation between sections. Moreover, the generated text typically does not follow a coherent logic and often lacks a proper conclusion.


## 🎯 0-shot, one-shot, few-shot learning: comparison

In [ ]:
def generate_comparison_for_title(title, model, tokenizer, examples=[], max_new_tokens=1024):
    device = model.device

    def generate_from_prompt(prompt, max_new_tokens):
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        # Usa max_new_tokens
        output = model.generate(
            inputs['input_ids'],
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        return tokenizer.decode(output[0], skip_special_tokens=True)

    # Zero-shot: solo il titolo e la sezione da completare
    zero_prompt = f"Generate a new recipe with this TITLE: {title}\n\nINGREDIENTS:\n"
    zero_shot_output = generate_from_prompt(zero_prompt, max_new_tokens)

    # One-shot
    if len(examples) >= 1:
        one_prompt = examples[0].strip() + "\n\n" + f"Generate a new recipe with this TITLE: {title}\n\nINGREDIENTS:\n"
        one_shot_output = generate_from_prompt(one_prompt, max_new_tokens)
    else:
        one_shot_output = "(Not enough examples provided)"

    # Few-shot (2)
    if len(examples) >= 2:
        few_prompt = "\n\n".join([ex.strip() for ex in examples[:2]]) + "\n\n" + f"Generate a new recipe with this TITLE: {title}\n\nINGREDIENTS:\n"
        few_shot_output = generate_from_prompt(few_prompt, max_new_tokens)
    else:
        few_shot_output = "(Not enough examples provided)"

    # Output ben formattato
    print("="*80)
    print(f"🍽 TITLE: {title}\n")
    print("🔹 Zero-shot:\n" + zero_shot_output + "\n")
    print("-"*80)
    print("🔹 One-shot:\n" + one_shot_output + "\n")
    print("-"*80)
    print("🔹 Few-shot:\n" + few_shot_output + "\n")
    print("="*80 + "\n\n")

In [ ]:
test_titles = [
    "Chocolate Chip Pancakes",
    # "Lemon Garlic Roasted Chicken",
    # "Vegan Strawberry Smoothie"
]
examples = [
    "TITLE: scrambled eggs\nINGREDIENTS:\n- 2 eggs\n- 1 tbsp butter\n\nDIRECTIONS:\n1. Beat the eggs.\n2. Cook in butter.",
    "TITLE: garlic pasta\nINGREDIENTS:\n- 200g spaghetti\n- 2 garlic cloves\n- 2 tbsp olive oil\n- Salt\n\nDIRECTIONS:\n1. Cook the spaghetti in salted water.\n2. Sauté garlic in olive oil.\n3. Mix pasta with garlic oil."
]
for title in test_titles:
    generate_comparison_for_title(title, model, tokenizer, examples=examples)

🍽 TITLE: Chocolate Chip Pancakes

🔹 Zero-shot:
Generate a new recipe with this TITLE: Chocolate Chip Pancakes

INGREDIENTS:
.75 oz (16 g) white flour, divided; 1/2 cup sugar and 2 tablespoons granulated vegetable oil in large bowl or blender until well combined. Add remaining ingredients to the wok as needed for 8-10 minutes more than normal on low speed if desired before adding into batter . Cover dough loosely so that it is completely covered while you stir mixture through each time , then roll out half way round onto prepared surface using your hands at one end of bottom rack according by hand method described below after rolling over slightly from top down but keeping an eye off all edges when measuring .) Cut approximately 3" long pieces about 4 inches wide x 6" deep between middle circles making sure they are just touching once which will prevent them falling apart during kneading If necessary use disposable scissors cut ends separately instead The last step may require cutting l

### 🔍 Zero-shot vs One-shot vs Few-shot Comparison

From this initial experiment, we can draw several observations regarding the effectiveness of prompting strategies on the pre-trained GPT-2 model:

- **Zero-shot**: The output begins in a somewhat relevant way (mentioning flour, sugar, etc.), but it quickly derails into incoherent instructions and unstructured narrative. The ingredients and directions are jumbled together without any clear separation, and the overall result lacks logical consistency. This is expected from a pre-trained model that has not seen task-specific formatting.

- **One-shot**: With a single example, the model starts to imitate the expected structure a bit more. It recognizes the format for listing ingredients and directions, and attempts to follow a recipe-like tone. However, the content remains largely ungrounded, with verbose and rambling text. While slightly more readable, it still fails to produce a clean or realistic recipe.

- **Few-shot**: Despite having two structured examples, the output remains minimal and uninformative. The model appears to mimic some syntactic patterns, but the content is sparse and lacks any meaningful culinary instruction. Notice that few-shot learning is not suitable for GPT-2 model since the prompt starts exceeding the context window even with short examples, and it is not possible to increase the number of examples too.

Neither the one-shot nor the few-shot approaches are able to produce a clearly structured or useful recipe with the pre-trained GPT-2. While the one-shot variant shows slight improvement in format awareness, none of the generations are coherent or complete. This highlights the limitations of the base model and motivates the need for fine-tuning on domain-specific data.


## Prepare the fine tuning



### Dataset creation

Create a dataset for training.

In [ ]:
from torch.utils.data import Dataset
import torch

class RecipeDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size):
        print("Loading recipes...")
        with open(file_path, 'r') as f:
            text = f.read()

        # Split into recipes
        recipes = text.split("Generate a new recipe with this TITLE:")
        recipes = ["Generate a new recipe with this TITLE:" + recipe for recipe in recipes[1:]]
        print(f"Found {len(recipes)} recipes")

        # Tokenize all recipes with proper padding and truncation
        self.examples = []
        self.recipe_lengths = []
        self.average_len = 0
        for i, recipe in enumerate(recipes):
            if i % 10000 == 0:
                print(f"Tokenizing recipe {i}/{len(recipes)}")

            self.recipe_lengths .append(len(recipe))

            # Tokenize the recipe
            # Add proper padding and truncation parameters
            encoding = tokenizer(
                recipe,
                truncation=True,
                padding='max_length',
                max_length=block_size,
                return_tensors='pt'
            )

            # Remove the batch dimension that return_tensors='pt' adds
            example = {key: val.squeeze(0) for key, val in encoding.items()}

            # Add labels (same as input_ids for causal language modeling)
            example["labels"] = example["input_ids"].clone()

            self.examples.append(example)

        self.average_len = sum(self.recipe_lengths) / len(self.recipe_lengths)
        print(f"Created dataset with {len(self.examples)} examples")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

# Use your new dataset
train_dataset = RecipeDataset('recipes_train.txt', tokenizer, block_size=750)


Loading recipes...
Found 100000 recipes
Tokenizing recipe 0/100000
Tokenizing recipe 10000/100000
Tokenizing recipe 20000/100000
Tokenizing recipe 30000/100000
Tokenizing recipe 40000/100000
Tokenizing recipe 50000/100000
Tokenizing recipe 60000/100000
Tokenizing recipe 70000/100000
Tokenizing recipe 80000/100000
Tokenizing recipe 90000/100000
Created dataset with 100000 examples


In [ ]:
train_dataset.average_len

882.76491

In [ ]:
# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
)


### Set Up Training Arguments

Configure the training parameters.

In [ ]:
# Define training arguments
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    save_strategy="no",
    per_device_train_batch_size=8,
    logging_steps=100,
    logging_dir="logs",
    fp16=True,
    report_to="none",
)

## 🔁 Fine-tune the Model

Start the training process.

In [ ]:
# Initialize trainer and disable wandb
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

# Train the model
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.068500
200,1.926600
300,1.875100
400,1.877600
500,1.862900
600,1.832700
700,1.801100
800,1.804300
900,1.763100
1000,1.760700


TrainOutput(global_step=37500, training_loss=1.4762709110514323, metrics={'train_runtime': 36550.3254, 'train_samples_per_second': 8.208, 'train_steps_per_second': 1.026, 'total_flos': 1.148256e+17, 'train_loss': 1.4762709110514323, 'epoch': 3.0})

### Save the Fine-tuned Model

In [ ]:
# Save model and tokenizer
model_path = 'recipe_generator_model'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to {model_path}")

Model saved to recipe_generator_model


## 🧾 FINE-tuned model evaluation

Now we use the fine-tuned model to generate new recipes from titles and compare them with the reference recipes.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Path to your saved model
model_path = "recipe_generator_model"

# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded from {model_path} and moved to {device}")

Model loaded from recipe_generator_model and moved to cpu


In [ ]:
for id in eval_selected_recipes_id:
    title = recipes[id]['title']
    reference_text = format_recipe_for_fine_tuning(recipes[id])

    results.append(evaluate_generated_recipe(title, reference_text, model, tokenizer))


🍽 TITLE: Date Rolled Cookies

📜 REFERENCE RECIPE:
 Generate a new recipe with this TITLE: Date Rolled Cookies

TITLE: Date Rolled Cookies

INGREDIENTS:
- 1 c. butter
- 1 c. white sugar
- 1 c. brown sugar
- 1 tsp. salt
- 3 beaten eggs
- 1 Tbsp. vanilla
- 4 c. flour
- 1 1/2 tsp. soda
- 1 Tbsp. cold water

DIRECTIONS:
1. Mix the flour and soda together.
2. Roll 1/2 dough on floured board into a rectangle, 1/2-inch thick.
3. Spread filling all over top (then sprinkle with chopped English walnuts, if desired). Roll like a jelly roll and place in refrigerator.
4. When firm, slice and bake 10 to 12 minutes at 350° or 10 minutes at 375° (top of cookie may be sprinkled with sugar and top with 1/2 walnut before baking).

 ...

----------------------------------------------------------------------------------------------------
🧠 GENERATED RECIPE:
 Generate a new recipe with this TITLE: Date Rolled Cookies

INGREDIENTS:
- 1/2 c. shortening or butter, softened (2 sticks) and cooled slightly
- 2/3 c

Compared to the outputs of the pre-trained GPT-2 model, the fine-tuned model demonstrates significant improvements across multiple dimensions:

- The fine-tuned model correctly understands the prompt "Generate a new recipe with this TITLE" and reliably produces content that reflects the intended purpose.

- While the pre-trained model struggled to consistently separate ingredients from directions—and often blended formatting or hallucinated irrelevant sections—the fine-tuned model now generates recipes with a clearer, more recognizable structure. Ingredients and instructions are typically well-defined, and the flow of the recipe more closely matches real-world examples.

- Generated ingredient lists are more plausible and complete, often resembling standard culinary proportions and common ingredients. Instructions, while not always perfect, now include logical steps and follow a consistent cooking process—unlike the disorganized or nonsensical procedures seen in the pre-trained outputs.

Despite some lingering imperfections—such as occasional verbosity or repetition—the fine-tuned model offers a substantial qualitative leap. This validates the effectiveness of domain-specific fine-tuning.



In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df['model'] = ["Pretrained", "Pretrained", "Pretrained", "Fine-tuned", "Fine-tuned", "Fine-tuned"]
df = df.sort_values(by=["title"]).reset_index(drop=True)
df_summary = df[["model", "title", "semantic_similarity", "rouge1_f1", "meteor"]]
df_summary = df_summary.round(4)
df_summary

,model,title,semantic_similarity,rouge1_f1,meteor
0,Pretrained,"""Cheaters"" Mexican Bake",0.6971,0.1782,0.2513
1,Fine-tuned,"""Cheaters"" Mexican Bake",0.8335,0.1772,0.3015
2,Pretrained,Date Rolled Cookies,0.7714,0.3308,0.2103
3,Fine-tuned,Date Rolled Cookies,0.8272,0.2522,0.3193
4,Pretrained,Icebox Fruit Cake,0.7935,0.2591,0.2380
5,Fine-tuned,Icebox Fruit Cake,0.7617,0.1289,0.2360



Also observing a quantitative comparison of pre-trained vs fine-tuned GPT-2, we can clearly confirm the **effectiveness of domain-specific fine-tuning** in recipe generation. Key observations:

- **Semantic Similarity**: In all three recipes, the fine-tuned model consistently achieves **higher SBERT cosine similarity**, indicating that the generated recipes better preserve the meaning and thematic coherence of the reference recipe titles and their content.

- **METEOR Score**: The fine-tuned model outperforms the base model in all three cases, demonstrating improvements in grammar, word order, synonym usage, and overall fluency. This supports the qualitative observation that fine-tuning improves naturalness and readability.

- **ROUGE-1 F1 Score**: In two out of three cases, the ROUGE-1 F1 score is **lower** for the fine-tuned model. This might seem counterintuitive but is not necessarily a negative signal. ROUGE heavily rewards surface-level n-gram overlaps, while fine-tuned models tend to generate more diverse, paraphrased, or semantically faithful responses that may diverge in wording. This reflects a *trade-off between lexical overlap and content originality*.

